### Transformar Datos Job Positions
1. Preprocesar la cadena JSON para corregir los problemas de calidad de los datos
2. Transformar cadena JSON en objeto JSON
3. Escribir los datos "transformados" en el schema "silver" - "job_positions_json"
4. Acceder a los elementos del objeto JSON
5. Eliminar elementos duplicados del array
6. Anular el anidamiento del array
7. Escribir los datos "transformados" en el schema "silver" - "job_positions"

In [0]:
SELECT * FROM zenviro.bronze.v_job_positions;

#### 1. Preprocesar la cadena JSON para corregir los problemas de calidad de los datos

In [0]:
SELECT value,
       regexp_replace(value, '"title": ([^",}]+)', '"title": "\$1"') AS fixed_value
FROM zenviro.bronze.v_job_positions;

In [0]:
CREATE OR REPLACE TEMP VIEW tv_job_positions_fixed
AS
  SELECT value,
        regexp_replace(value, '"title": ([^",}]+)', '"title": "\$1"') AS fixed_value
  FROM zenviro.bronze.v_job_positions;

#### 2. Transformar cadena JSON en objeto JSON

In [0]:
SELECT schema_of_json(fixed_value) AS schema,
        fixed_value
FROM tv_job_positions_fixed
LIMIT 1;

In [0]:
SELECT from_json(fixed_value,
                  'STRUCT<description: STRING, job_position_id: BIGINT, salary: ARRAY<STRUCT<salary_max: BIGINT, salary_min: BIGINT>>, title: STRING>') AS json_value,
        fixed_value
FROM tv_job_positions_fixed;

#### 3. Escribir los datos "transformados" en el schema "silver" - "job_positions_json"

In [0]:
DROP TABLE IF EXISTS zenviro.silver.job_positions_json;
CREATE TABLE IF NOT EXISTS zenviro.silver.job_positions_json
AS
  SELECT from_json(fixed_value,
                    'STRUCT<description: STRING, job_position_id: BIGINT, salary: ARRAY<STRUCT<salary_max: BIGINT, salary_min: BIGINT>>, title: STRING>') AS json_value,
          fixed_value
  FROM tv_job_positions_fixed;

In [0]:
SELECT * FROM zenviro.silver.job_positions_json;

#### 4. Acceder a los elementos del objeto JSON
> `<column_name.object>`

In [0]:
SELECT json_value.job_position_id,
       json_value.title,
       json_value.description,
       json_value.salary
FROM zenviro.silver.job_positions_json;

#### 5. Eliminar elementos duplicados del array
[Doc - Función "array_distinct"](https://learn.microsoft.com/es-es/azure/databricks/sql/language-manual/functions/array_distinct)

In [0]:
SELECT json_value.job_position_id,
       json_value.title,
       json_value.description,
       array_distinct(json_value.salary) as salary
FROM zenviro.silver.job_positions_json;

#### 6. Anular el anidamiento del array

In [0]:
CREATE OR REPLACE TEMP VIEW tv_job_positions_explode
AS
  SELECT json_value.job_position_id,
        json_value.title,
        json_value.description,
        explode(array_distinct(json_value.salary)) as salary
  FROM zenviro.silver.job_positions_json;

In [0]:
SELECT * FROM tv_job_positions_explode;

In [0]:
SELECT job_position_id,
       title,
       description,
       salary.salary_min,
       salary.salary_max
FROM tv_job_positions_explode;

#### 7. Escribir los datos "transformados" en el schema "silver" - "job_positions"

In [0]:
CREATE TABLE IF NOT EXISTS zenviro.silver.job_positions
AS
  SELECT job_position_id,
        title,
        description,
        salary.salary_min,
        salary.salary_max
  FROM tv_job_positions_explode;

In [0]:
SELECT * FROM zenviro.silver.job_positions;